In [1]:
import pandas as pd
import os

print("--- Downloading NHANES 2017-2018 Modules ---")

# CDC URLs for Demographics, Body Measures, Sleep, Diabetes, and Blood Pressure
urls = {
    "DEMO": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.XPT",
    "BMX":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BMX_J.XPT",
    "SLQ":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/SLQ_J.XPT",
    "DIQ":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DIQ_J.XPT",
    "BPQ":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPQ_J.XPT"
}

dfs = {}
for name, url in urls.items():
    print(f"Fetching {name}...")
    # pandas read_sas handles the CDC's .XPT files natively
    dfs[name] = pd.read_sas(url)

# Merge all modules on 'SEQN' (Participant ID)
print("\n--- Merging on SEQN ---")
df_nhanes = dfs["DEMO"]
for name in ["BMX", "SLQ", "DIQ", "BPQ"]:
    df_nhanes = pd.merge(df_nhanes, dfs[name], on='SEQN', how='inner')

# Save locally to save time later
output_dir = os.path.join("..", "data", "raw")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "nhanes_merged.csv")
df_nhanes.to_csv(output_path, index=False)

print(f"\n✅ Merged Dataset Saved to: {output_path}")
print(f"Dataset Shape: {df_nhanes.shape}")

--- Downloading NHANES 2017-2018 Modules ---
Fetching DEMO...
Fetching BMX...
Fetching SLQ...
Fetching DIQ...
Fetching BPQ...

--- Merging on SEQN ---

✅ Merged Dataset Saved to: ..\data\raw\nhanes_merged.csv
Dataset Shape: (5828, 139)


In [2]:
# Cell 2: Inspect raw target responses according to CDC codebooks
print("--- Raw DIQ010 (Doctor told you have diabetes) ---")
print(df_nhanes['DIQ010'].value_counts(dropna=False))

print("\n--- Raw BPQ020 (Ever told you had high blood pressure) ---")
print(df_nhanes['BPQ020'].value_counts(dropna=False))

--- Raw DIQ010 (Doctor told you have diabetes) ---
DIQ010
2.0    4811
1.0     843
3.0     170
9.0       4
Name: count, dtype: int64

--- Raw BPQ020 (Ever told you had high blood pressure) ---
BPQ020
2.0    3788
1.0    2030
9.0      10
Name: count, dtype: int64


In [3]:
# 1. Filter valid adult cohort with definitive Yes/No answers
df_cohort = df_nhanes[df_nhanes['RIDAGEYR'] >= 20].copy()
df_cohort = df_cohort[df_cohort['DIQ010'].isin([1, 2])]
df_cohort = df_cohort[df_cohort['BPQ020'].isin([1, 2])]

# 2. Binary target mapping (1 = Yes, 0 = No)
df_cohort['target_diabetes'] = (df_cohort['DIQ010'] == 1).astype(int)
df_cohort['target_hypertension'] = (df_cohort['BPQ020'] == 1).astype(int)

# 3. Check availability of primary general and sleep features
check_cols = [
    'RIDAGEYR',  # Age
    'RIAGENDR',  # Gender
    'BMXBMI',    # Body Mass Index
    'BMXWAIST',  # Waist Circumference
    'SLD012',    # Sleep duration (weekdays) in hours (Cycle J)
    'SLQ050'     # Ever told doctor had trouble sleeping (1=Yes, 2=No)
]

# Find matching columns present in the dataframe
present_cols = [col for col in check_cols if col in df_cohort.columns]

print(f"Cohort Size: {df_cohort.shape[0]} participants")
print("\nTarget Breakdown:")
print("Diabetes (0 = No, 1 = Yes):")
print(df_cohort['target_diabetes'].value_counts())
print("\nHypertension (0 = No, 1 = Yes):")
print(df_cohort['target_hypertension'].value_counts())

print("\nKey Feature Missing Values:")
print(df_cohort[present_cols].isnull().sum())

Cohort Size: 5088 participants

Target Breakdown:
Diabetes (0 = No, 1 = Yes):
target_diabetes
0    4257
1     831
Name: count, dtype: int64

Hypertension (0 = No, 1 = Yes):
target_hypertension
0    3163
1    1925
Name: count, dtype: int64

Key Feature Missing Values:
RIDAGEYR      0
RIAGENDR      0
BMXBMI       88
BMXWAIST    318
SLD012       43
SLQ050        0
dtype: int64


In [4]:
# 1. Clean feature representations
df_final = df_cohort.copy()

# Recode Gender: 1 (Male) -> 0, 2 (Female) -> 1
df_final['gender_female'] = (df_final['RIAGENDR'] == 2).astype(int)

# Recode Trouble Sleeping (SLQ050): 1 (Yes) -> 1, 2 (No) -> 0
df_final['trouble_sleeping'] = (df_final['SLQ050'] == 1).astype(int)

# Rename core features for clarity in thesis tables and Streamlit UI
df_final = df_final.rename(columns={
    'RIDAGEYR': 'age',
    'BMXBMI': 'bmi',
    'BMXWAIST': 'waist_circumference',
    'SLD012': 'sleep_duration_hrs'
})

# 2. Select final feature set + targets (Zero leakage columns)
selected_columns = [
    # Demographics & Anthropometrics (Model A Baseline)
    'age',
    'gender_female',
    'bmi',
    'waist_circumference',
    # Sleep Variables (Added in Model B)
    'sleep_duration_hrs',
    'trouble_sleeping',
    # Targets
    'target_diabetes',
    'target_hypertension'
]

df_final = df_final[selected_columns]

# 3. Save to data/processed
output_path = os.path.join("..", "data", "processed", "nhanes_processed.csv")
df_final.to_csv(output_path, index=False)

print("--- NHANES Curation Complete ---")
print(f"File saved to: {output_path}")
print(f"Final Processed Shape: {df_final.shape}")
print("\nFeature Summary Statistics:")
print(df_final.describe().T[['count', 'mean', 'min', 'max']])


--- NHANES Curation Complete ---
File saved to: ..\data\processed\nhanes_processed.csv
Final Processed Shape: (5088, 8)

Feature Summary Statistics:
                      count        mean   min    max
age                  5088.0   51.180818  20.0   80.0
gender_female        5088.0    0.517885   0.0    1.0
bmi                  5000.0   29.800180  14.2   86.2
waist_circumference  4770.0  100.631216  57.9  169.5
sleep_duration_hrs   5045.0    7.590585   2.0   14.0
trouble_sleeping     5088.0    0.280464   0.0    1.0
target_diabetes      5088.0    0.163325   0.0    1.0
target_hypertension  5088.0    0.378341   0.0    1.0
